# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

Colab의 GPU 환경에서 개발되었습니다.
- 런타임 - 런타임 유형 변경 - GPU로 변경(T4 GPU 등)



# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작

In [1]:
!pip -q install git+https://github.com/huggingface/transformers accelerate
!pip -q install --upgrade Pillow
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" datasets pillow pandas --upgrade

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

Torch version: 2.10.0+cu128
CUDA version: 12.8
cuDNN version: 91002


# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

본 베이스라인은 colab에서 구글 드라이브를 마운트하여 사용합니다.

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

#### 실습 참고 내용

    챕터 2-2 합성 데이터 실습
    - 구글 드라이브 마운트 : drive()

In [3]:
# 구글드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
# 압축 해제
!unzip "/content/drive/My Drive/060401-ai-challenge.zip" -d "/content/"

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: /content/train/train_0074.jpg  
  inflating: /content/train/train_0075.jpg  
  inflating: /content/train/train_0076.jpg  
  inflating: /content/train/train_0077.jpg  
  inflating: /content/train/train_0078.jpg  
  inflating: /content/train/train_0079.jpg  
  inflating: /content/train/train_0080.jpg  
  inflating: /content/train/train_0081.jpg  
  inflating: /content/train/train_0082.jpg  
  inflating: /content/train/train_0083.jpg  
  inflating: /content/train/train_0084.jpg  
  inflating: /content/train/train_0085.jpg  
  inflating: /content/train/train_0086.jpg  
  inflating: /content/train/train_0087.jpg  
  inflating: /content/train/train_0088.jpg  
  inflating: /content/train/train_0089.jpg  
  inflating: /content/train/train_0090.jpg  
  inflating: /content/train/train_0091.jpg  
  inflating: /content/train/train_0092.jpg  
  inflating: /content/train/train_0093.jpg  
  inflating: /content/train/train_0094.jpg  
  inflating: /conte

# 라이브러리, 데이터, 설정

In [4]:
import os, random, math, json, gc, re
from dataclasses import dataclass
from contextlib import nullcontext
from typing import Any

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None

DATA_ROOT  = "/content"
TRAIN_CSV  = os.path.join(DATA_ROOT, "train.csv")
TEST_CSV   = os.path.join(DATA_ROOT, "test.csv")
AUDIT_DIR  = os.path.join(DATA_ROOT, "audit")
os.makedirs(AUDIT_DIR, exist_ok=True)

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
IMAGE_SIZE = 384
MAX_NEW_TOKENS = 2
SEED       = 42
VAL_SIZE   = 0.10
BATCH_SIZE = 4        # 버그픽스: 1 → 4
GRAD_ACCUM = 2        # 버그픽스: 8 → 2 (effective batch 8 유지)
LR         = 1e-4
NUM_EPOCHS = 3        # 버그픽스: 1 → 5
EARLY_STOP_PATIENCE = 2
USE_4BIT   = False    # 버그픽스: H100이므로 False
CHECK_IMAGES = True

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

def resolve_path(p):
    p = str(p)
    return p if os.path.isabs(p) else os.path.join(DATA_ROOT, p)

train_df["abs_path"] = train_df["path"].map(resolve_path)
test_df["abs_path"]  = test_df["path"].map(resolve_path)

def find_hard_errors(df, has_answer=True, check_images=True):
    issue_frames = []
    base_cols = ["id","path","abs_path","question","a","b","c","d"]
    if has_answer:
        base_cols.append("answer")

    # 1) 결측치
    missing_mask = df[base_cols].isna().any(axis=1)
    if missing_mask.any():
        tmp = df.loc[missing_mask, base_cols].copy()
        tmp["issue"] = "missing_value"
        issue_frames.append(tmp)

    # 2) 유효하지 않은 answer
    if has_answer:
        invalid_mask = ~df["answer"].astype(str).str.strip().str.lower().isin(list("abcd"))
        if invalid_mask.any():
            tmp = df.loc[invalid_mask, base_cols].copy()
            tmp["issue"] = "invalid_answer"
            issue_frames.append(tmp)

    # 3) 보기 중복 (train_4196 등)
    dup_mask = df[["a","b","c","d"]].astype(str).apply(
        lambda row: len({x.strip() for x in row.tolist()}) < 4, axis=1
    )
    if dup_mask.any():
        tmp = df.loc[dup_mask, base_cols].copy()
        tmp["issue"] = "duplicate_options"
        issue_frames.append(tmp)

    # 4) 파일 없음
    no_file_mask = ~df["abs_path"].map(os.path.exists)
    if no_file_mask.any():
        tmp = df.loc[no_file_mask, base_cols].copy()
        tmp["issue"] = "missing_file"
        issue_frames.append(tmp)

    # 5) 깨진 이미지
    if check_images:
        broken = []
        for idx in tqdm(df.index[~no_file_mask].tolist(), desc="Image audit"):
            try:
                with Image.open(df.at[idx, "abs_path"]) as img:
                    img.verify()
            except Exception:
                broken.append(idx)
        if broken:
            tmp = df.loc[broken, base_cols].copy()
            tmp["issue"] = "broken_image"
            issue_frames.append(tmp)

    if issue_frames:
        issues = pd.concat(issue_frames, ignore_index=True).drop_duplicates(subset=["id","issue"])
    else:
        issues = pd.DataFrame(columns=base_cols + ["issue"])

    bad_ids  = set(issues["id"].tolist())
    clean_df = df[~df["id"].isin(bad_ids)].copy().reset_index(drop=True)
    return clean_df, issues

train_clean_df, train_issue_df = find_hard_errors(
    train_df, has_answer=True, check_images=CHECK_IMAGES
)

train_issue_df.to_csv(os.path.join(AUDIT_DIR, "train_hard_errors.csv"), index=False)

audit_summary = {
    "raw": int(len(train_df)),
    "clean": int(len(train_clean_df)),
    "removed": int(len(train_df) - len(train_clean_df)),
    "issues": train_issue_df["issue"].value_counts().to_dict() if len(train_issue_df) else {},
    "answer_dist": train_clean_df["answer"].value_counts().sort_index().to_dict(),
}
print(json.dumps(audit_summary, ensure_ascii=False, indent=2))

# stratified split
train_subset, valid_subset = train_test_split(
    train_clean_df,
    test_size=VAL_SIZE,
    random_state=SEED,
    shuffle=True,
    stratify=train_clean_df["answer"],
)
train_subset = train_subset.reset_index(drop=True)
valid_subset = valid_subset.reset_index(drop=True)

train_subset.to_csv(os.path.join(AUDIT_DIR, "train_split.csv"), index=False)
valid_subset.to_csv(os.path.join(AUDIT_DIR, "valid_split.csv"), index=False)

print(f"train: {len(train_subset)}, valid: {len(valid_subset)}")
print(f"train 정답 분포:\n{train_subset['answer'].value_counts().sort_index()}")
print(f"valid 정답 분포:\n{valid_subset['answer'].value_counts().sort_index()}")

Device: cuda


Image audit:   0%|          | 0/5073 [00:00<?, ?it/s]

{
  "raw": 5073,
  "clean": 5072,
  "removed": 1,
  "issues": {
    "duplicate_options": 1
  },
  "answer_dist": {
    "a": 1262,
    "b": 1265,
    "c": 1311,
    "d": 1234
  }
}
train: 4564, valid: 508
train 정답 분포:
answer
a    1136
b    1138
c    1180
d    1110
Name: count, dtype: int64
valid 정답 분포:
answer
a    126
b    127
c    131
d    124
Name: count, dtype: int64


# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [5]:
# BF16으로 변경, r=8 그대로 유지
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

# 버그픽스: H100이므로 BF16, 4bit 안 씀
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,  # ← 수정
    trust_remote_code=True,
).to(device)

base_model.gradient_checkpointing_enable({"use_reentrant": False})
base_model.config.use_cache = False

lora_config = LoraConfig(
    r=8,           # E0에서는 그대로 유지 (E2에서 r=64로 실험)
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

trainable params: 18,576,384 || all params: 3,773,199,360 || trainable%: 0.4923


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [6]:
SYSTEM_INSTRUCT = (
    "당신은 재활용품 이미지 분석 전문가입니다. "
    "이미지에서 색상, 재질, 개수, 형태를 주의 깊게 관찰하세요. "
    "주어진 선택지 중 이미지와 가장 정확히 일치하는 하나를 고르세요. "
    "반드시 a, b, c, d 중 하나의 소문자 알파벳만 출력하세요. 다른 설명은 절대 금지입니다."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"질문: {question}\n\n"
        f"a. {a}\n"
        f"b. {b}\n"
        f"c. {c}\n"
        f"d. {d}\n\n"
        "정답:"
    )

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [7]:
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df        = df.reset_index(drop=True)
        self.processor = processor
        self.train     = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["abs_path"]).convert("RGB")

        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        gold = str(row["answer"]).strip().lower() if "answer" in row else None

        user_text = build_mc_prompt(str(row["question"]), a, b, c, d)

        prompt_messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user",   "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": user_text},
            ]},
        ]

        item = {
            "prompt_messages": prompt_messages,
            "image": img,
            "gold":  gold,
        }

        if self.train:
            item["full_messages"] = prompt_messages + [
                {"role": "assistant",
                 "content": [{"type": "text", "text": gold}]}
            ]

        return item


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        full_texts, prompt_texts, images = [], [], []

        for sample in batch:
            full_texts.append(
                self.processor.apply_chat_template(
                    sample["full_messages"],
                    tokenize=False,
                    add_generation_prompt=False,
                )
            )
            prompt_texts.append(
                self.processor.apply_chat_template(
                    sample["prompt_messages"],
                    tokenize=False,
                    add_generation_prompt=True,
                )
            )
            images.append(sample["image"])

        enc = self.processor(
            text=full_texts, images=images,
            padding=True, return_tensors="pt",
        )
        prompt_enc = self.processor(
            text=prompt_texts, images=images,
            padding=True, return_tensors="pt",
        )

        labels = enc["input_ids"].clone()
        labels[enc["attention_mask"] == 0] = -100
        prompt_lens = prompt_enc["attention_mask"].sum(dim=1).tolist()
        for i, pl in enumerate(prompt_lens):
            labels[i, :pl] = -100

        enc["labels"] = labels
        return enc

# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [8]:

# stratified split 결과 사용, batch=4
train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds  = VQAMCDataset(valid_subset, processor, train=True)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,   # 버그픽스: 1 → 4
    shuffle=True,
    num_workers=2,            # 버그픽스: 0 → 2
    collate_fn=DataCollator(processor, True),
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    collate_fn=DataCollator(processor, True),
)

print(f"train batches: {len(train_loader)}, valid batches: {len(valid_loader)}")

train batches: 1141, valid batches: 127


# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
def extract_choice(text: str) -> str:
    text = str(text).strip().lower()
    if text in {"a","b","c","d"}:
        return text
    match = re.findall(r"[abcd]", text)
    return match[0] if match else "a"

@torch.no_grad()
def predict_one(model, processor, row, device):
    img = Image.open(row["abs_path"]).convert("RGB")
    user_text = build_mc_prompt(
        str(row["question"]),
        str(row["a"]), str(row["b"]),
        str(row["c"]), str(row["d"]),
    )
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user",   "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text": user_text},
        ]},
    ]
    text   = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    amp_ctx = torch.cuda.amp.autocast(dtype=torch.float16) if device == "cuda" else nullcontext()
    with amp_ctx:
        out_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    gen_ids   = out_ids[:, inputs["input_ids"].shape[1]:]
    pred_text = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
    return extract_choice(pred_text), pred_text

@torch.no_grad()
def evaluate_generation_accuracy(model, processor, valid_df, device):
    refs, preds, raws = [], [], []
    for _, row in tqdm(valid_df.iterrows(), total=len(valid_df), desc="valid-acc"):
        pred, raw = predict_one(model, processor, row, device)
        preds.append(pred)
        raws.append(raw)
        refs.append(str(row["answer"]).strip().lower())
    acc = float(np.mean(np.array(preds) == np.array(refs)))
    return acc, preds, raws, refs

# 옵티마이저
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
num_update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
num_training_steps = NUM_EPOCHS * num_update_steps_per_epoch
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(num_training_steps * 0.03)),
    num_training_steps=num_training_steps,
)

# 학습
history        = []
best_val_acc   = -1.0
best_epoch     = 0
patience_counter = 0
SAVE_DIR       = "/content/drive/My Drive/qwen2_5_vl_3b_lora_best"
os.makedirs(SAVE_DIR, exist_ok=True)
model = model.to(device)

for epoch in range(1, NUM_EPOCHS + 1):

    # Train
    model.train()
    optimizer.zero_grad(set_to_none=True)
    train_loss_sum, train_steps = 0.0, 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [train]")
    for step, batch in enumerate(pbar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            loss = model(**batch).loss / GRAD_ACCUM
        loss.backward()
        train_loss_sum += loss.item() * GRAD_ACCUM
        train_steps    += 1

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        pbar.set_postfix(train_loss=f"{train_loss_sum / train_steps:.4f}")

    train_loss = train_loss_sum / max(1, train_steps)

    # Valid Loss
    model.eval()
    val_loss_sum, val_steps = 0.0, 0
    with torch.no_grad():
        for batch in tqdm(valid_loader, desc=f"Epoch {epoch} [val-loss]"):
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.cuda.amp.autocast(dtype=torch.float16):
                val_loss_sum += model(**batch).loss.item()
            val_steps += 1
    val_loss = val_loss_sum / max(1, val_steps)

    # Valid Accuracy
    val_acc, val_preds, val_raws, val_refs = evaluate_generation_accuracy(
        model, processor, valid_subset, device
    )

    # 기록
    history.append({
        "epoch": epoch, "train_loss": round(train_loss, 4),
        "val_loss": round(val_loss, 4), "val_acc": round(val_acc, 4),
    })
    pd.DataFrame(history).to_csv("/content/drive/My Drive/training_history.csv", index=False)

    val_pred_df = valid_subset[["id","question","a","b","c","d","answer"]].copy()
    val_pred_df["pred"]       = val_preds
    val_pred_df["raw_output"] = val_raws
    val_pred_df["correct"]    = val_pred_df["answer"] == val_pred_df["pred"]
    val_pred_df.to_csv(f"/content/drive/My Drive/val_predictions_epoch_{epoch}.csv", index=False)

    print(f"\n[E0 Epoch {epoch}] train_loss={train_loss:.4f} | "
          f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

    # Best 저장 & Early stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch   = epoch
        patience_counter = 0
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)
        print(f"  ✅ best model 저장 (epoch={best_epoch}, val_acc={best_val_acc:.4f})")
    else:
        patience_counter += 1
        print(f"  ⏳ patience {patience_counter}/{EARLY_STOP_PATIENCE}")

    gc.collect()
    torch.cuda.empty_cache()

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"🛑 Early stopping @ epoch {epoch}")
        break

print(f"\n🏁 E0 완료 | best epoch={best_epoch}, val_acc={best_val_acc:.4f}")
print("train 전체 추론 시작...")
model.eval()
wrong_train = []
for _, row in tqdm(train_clean_df.iterrows(), total=len(train_clean_df), desc="train-check"):
    pred, raw = predict_one(model, processor, row, device)
    gold = str(row["answer"]).strip().lower()
    if pred != gold:
        wrong_train.append({
            "id": row["id"], "path": row["path"],
            "question": row["question"],
            "answer": gold, "pred": pred,
            "a": row["a"], "b": row["b"],
            "c": row["c"], "d": row["d"],
        })

wrong_df = pd.DataFrame(wrong_train)
wrong_df.to_csv("/content/drive/My Drive/train_wrong.csv", index=False)
print(f"의심 샘플: {len(wrong_df)}개 / {len(train_clean_df)}개")


Epoch 1/3 [train]:   0%|          | 0/1141 [00:00<?, ?it/s]

/tmp/ipykernel_29528/1612571131.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch 1 [val-loss]:   0%|          | 0/127 [00:00<?, ?it/s]

/tmp/ipykernel_29528/1612571131.py:102: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


valid-acc:   0%|          | 0/508 [00:00<?, ?it/s]

/tmp/ipykernel_29528/1612571131.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  amp_ctx = torch.cuda.amp.autocast(dtype=torch.float16) if device == "cuda" else nullcontext()
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



[E0 Epoch 1] train_loss=nan | val_loss=nan | val_acc=0.2480
  ✅ best model 저장 (epoch=1, val_acc=0.2480)


Epoch 2/3 [train]:   0%|          | 0/1141 [00:00<?, ?it/s]

/tmp/ipykernel_29528/1612571131.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
# 데이터 파서 : 모델의 응답에서 선지를 추출
def extract_choice(text: str) -> str:
    text = text.strip().lower()

    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last

    tokens = last.split()
    for tok in tokens:
        if tok in ["a", "b", "c", "d"]:
            return tok
    return "a"

# 추론을 위해 모든 레이어 활성화
model.eval()
preds = []

# 추론 루프
for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    img = Image.open(row["abs_path"]).convert("RGB")
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[
            {"type":"image","image":img},
            {"type":"text","text":user_text}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=2, do_sample=False,
                                 eos_token_id=processor.tokenizer.eos_token_id)
    output_text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    # print("output_text:", output_text)
    # print("extract_choice:", extract_choice(output_text))
    preds.append(extract_choice(output_text))

# 제출 파일 생성
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission.csv", index=False)
print("Saved /content/submission.csv")

In [ ]:
# 모델 응답 예시
print(output_text)